# AI Agent Demo

Minimal LangChain proof of concept that loads tools from the asyncroscopy MCP server and asks the model to use one or two of them. This will be useful to test our MCP server and new tools.

## Prereqs

Start the Tango stack on the microscope computer. Start MCP and the LLM device on the LLM computer. Both MCP and the LLM device use the microscope computer's Tango database:

```bash
uv run startup_scripts/run_servers.py
uv run startup_scripts/run_mcp.py --yaml configs/mcp.yaml
uv run startup_scripts/run_llm.py --yaml configs/gemma-llm.yaml
```

If not already done, install the optional AI extras:

```bash
uv sync --extra agent
# for local models do uv sync --extra agent --extra localagent
```

## Imports

In [34]:
import json
import os

import tango
from tiled.client import from_uri

## Ping Servers

In [44]:
MICROSCOPE_HOST = "localhost"
DB_PORT = 9094

os.environ["TANGO_HOST"] = f"{MICROSCOPE_HOST}:{DB_PORT}"
print(os.environ["TANGO_HOST"])

microscope = tango.DeviceProxy("asyncroscopy/instrument/default")
data = tango.DeviceProxy("asyncroscopy/data/default")
llm = tango.DeviceProxy("asyncroscopy/llm/default")

for proxy in (microscope, data, llm):
    proxy.set_timeout_millis(120_000)
    proxy.ping()
    print(proxy.name(), proxy.state())

tiled_config = json.loads(data.get_config())
client = from_uri(tiled_config["uri"])
print("Tiled:", tiled_config["uri"])

localhost:9094
asyncroscopy/instrument/default ON
asyncroscopy/data/default ON
asyncroscopy/llm/default ON
Tiled: http://localhost:9091


## Give a prompt

In [48]:
# adjust llm.max_steps if the LLM cannot complete a task
prompt = "Acquire a scanned HAADF image."
response = llm.query(prompt)
print(response)

I was unable to acquire the scanned HAADF image because the command timed out, and subsequent checks on the scan system (`SCAN_Status` and `SCAN_State`) failed with unknown exceptions. This suggests that the scanning hardware or its control service is currently unavailable.
